### **Knowledge DIstillation**

In [ ]:
!pip install --upgrade datasets fsspec transformers
!pip install transformers accelerate bitsandbytes

In [ ]:
import torch
import warnings
import time
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, get_scheduler, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

In [ ]:
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(images[i].squeeze(), cmap='gray')
    ax.set_title(f"Label: {labels[i].item()}")
    ax.axis('off')

plt.show()

In [ ]:
class TeacherMLP(nn.Module):
  def __init__(self, hidden1=512, hidden2=256):
    super().__init__()
    self.net = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28, hidden1),
        nn.ReLU(),
        nn.Linear(hidden1, hidden2),
        nn.ReLU(),
        nn.Linear(hidden2, 10)
    )

  def forward(self, x):
    return self.net(x)

teacher = TeacherMLP(hidden1=512, hidden2=256)
print(teacher)

In [ ]:
def train_teacher(model, train_loader, epochs, lr=1e-3):
  optimizer = optim.Adam(model.parameters(), lr=lr)
  loss_fn = nn.CrossEntropyLoss()
  model.train()

  for epoch in range(epochs):
    total_loss = 0
    for x, y in train_loader:
      optimizer.zero_grad()
      out = model(x)
      loss = loss_fn(out, y)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
    print(f"Teacher Epoch: {epoch}, Loss: {total_loss/len(train_loader):.4f}")

train_teacher(teacher, train_loader, epochs=1)
print(teacher.state_dict())

In [ ]:
class StudentMLP(nn.Module):
  def __init__(self, hidden1=128):
    super().__init__()
    self.net = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28, hidden1),
        nn.ReLU(),
        nn.Linear(hidden1, 10)
    )
  def forward(self, x):
    return self.net(x)

student = StudentMLP(hidden1=128)
print(student)

In [ ]:
def pretrain_student(student, train_loader, epochs=1, lr=1e-3):
  student.train()
  optimizer = optim.Adam(student.parameters(), lr=lr)
  loss_fn = nn.CrossEntropyLoss()
  for epoch in range(epochs):
    for x, y in train_loader:
      optimizer.zero_grad()
      out = student(x)
      loss = loss_fn(out, y)
      loss.backward()
      optimizer.step()

pretrain_student(student, train_loader, epochs=1)

In [ ]:
temperature = 2.0
alpha = 0.7
ce_loss = nn.CrossEntropyLoss()
kl_loss = nn.KLDivLoss(reduction="batchmean")
optimizer = optim.Adam(student.parameters(), lr=1e-3)

def distill(student, teacher, train_loader, epochs=1):
  for ep in range(epochs):
    student.train()
    total_loss = 0

    for x, y in train_loader:
      with torch.no_grad():
        teacher_logits = teacher(x)
        teacher_probs = torch.softmax(teacher_logits/temperature, dim=1)

      student_logits = student(x)
      student_logits_probs = torch.softmax(student_logits/temperature, dim=1)

      loss_soft = kl_loss(student_logits_probs, teacher_probs) * (temperature**2)
      loss_hard = ce_loss(student_logits, y)
      loss = alpha * loss_soft + (1-alpha) * loss_hard

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      total_loss += loss.item()
    print(f"Student Epoch: {ep}, Loss: {total_loss/len(train_loader):.4f}")

distill(student, teacher, train_loader)

In [ ]:
def evaluate(model, loader, name="Model"):
  model.eval()
  correct, total = 0, 0
  with torch.no_grad():
    for x, y in loader:
      out = model(x)
      pred = out.argmax(dim=1)
      correct += (pred == y).sum().item()
      total += y.size(0)
  accuracy = correct/total * 100
  print(f"{name} Accuracy: {correct/total:.2f}%")

evaluate(teacher, test_loader, "Teacher")
evaluate(student, test_loader, "Student")

In [ ]:
def predict(model, x):
  model.eval()
  with torch.no_grad():
    out = model(x)
    return out.argmax(dim=1)

sample_batch, sample_labels = next(iter(test_loader))
preds = predict(student, sample_batch)
torch.save(student.state_dict(), "distilled_student.pth")

print("Sample predictions (Student):", preds[:20])
print("True labels:", sample_labels[:20])
print("Student model saved as distilled_student.pth")

In [ ]:
batch_size   = 16
lr           = 5e-5
epochs       = 1
temperature  = 2.0
alpha_soft   = 0.5
max_len      = 128
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw = load_dataset("cardiffnlp/tweet_eval", "sentiment")
label_feature = raw["train"].features["label"]
train = raw['train'].shuffle(seed=42).select(range(2500))
val = raw['validation']
print("Label names:", label_feature.names)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(example):
  return tokenizer(example["text"], truncation=True, max_length=max_len)

tokenized = {}
tokenized["train"] = train.map(tokenize, batched=True, remove_columns=["text"])
tokenized["validation"] = val.map(tokenize, batched=True, remove_columns=["text"])

In [ ]:
num_labels = 3

collactor = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
train_loader = DataLoader(tokenized["train"], batch_size=batch_size, shuffle=True, collate_fn=collactor)
val_loader = DataLoader(tokenized["validation"], batch_size=batch_size,  shuffle=False, collate_fn=collactor)
teacher = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels).to(device)
student = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels).to(device)

for p in teacher.parameters():
  p.requires_grad = False
teacher.eval()

In [ ]:
ce_loss = nn.CrossEntropyLoss()
kl_loss = nn.KLDivLoss(reduction="batchmean")
optimizer = optim.Adam(student.parameters(), lr=lr)

lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=epochs * len(train_loader)
)

def distill_epoch():
  student.train()
  pbar = tqdm(train_loader)
  for batch in pbar:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    with torch.no_grad():
      teacher_logits = teacher(input_ids, attention_mask=attention_mask).logits
      teacher_soft = torch.softmax(teacher_logits/temperature, dim=1)

    student_logits = student(input_ids, attention_mask=attention_mask).logits
    student_soft = torch.log_softmax(student_logits/temperature, dim=1)

    loss_soft = kl_loss(student_soft, teacher_soft) * (temperature**2)
    loss_hard = ce_loss(student_logits, labels)
    loss = alpha_soft * loss_soft + (1-alpha_soft) * loss_hard

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    lr_scheduler.step()

    pbar.set_description(f"Loss: {loss.item():.4f}")

def evaluate():
  student.eval()
  correct = total = 0
  with torch.no_grad():
    for batch in val_loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)

      out = student(input_ids, attention_mask=attention_mask).logits
      pred = out.argmax(dim=1)
      correct += (pred == labels).sum().item()
      total += labels.size(0)

  return round(correct / total * 100, 2)

for ep in range(1, epochs + 1):
  distill_epoch()
  acc = evaluate()
  print(f"Epoch {ep}/{epochs} | Validation Accuracy: {acc}%")

In [ ]:
student.save_pretrained("distilled_student_model")
tokenizer.save_pretrained("distilled_student_model")

In [ ]:
def predict_and_evaluate(model, name, test_loader):
  model.eval()
  all_preds , all_labels = [], []
  start_time = time.time()

  with torch.no_grad():
    for batch in test_loader:
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)

      logits = model(input_ids, attention_mask=attention_mask).logits
      preds = logits.argmax(dim=1)

      all_preds.extend(preds.cpu().tolist())
      all_labels.extend(labels.cpu().tolist())

  total_time = time.time() - start_time
  acc = accuracy_score(all_labels, all_preds)
  avg_time = total_time / len(test_loader.dataset)

  print(f"\n {name}")
  print(f" Accuracy: {acc*100:.2f}%")
  print(f" Total Inference Time: {total_time:.2f} sec")
  print(f" Avg Time per Sample: {avg_time:.4f} sec")
  return acc, total_time, avg_time

In [ ]:
test = load_dataset("cardiffnlp/tweet_eval", "sentiment", split="test[:500]")
tokenized_test = test.map(tokenize, batched=True, remove_columns=["text"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dl = DataLoader(tokenized_test, batch_size=batch_size, shuffle=False, collate_fn=collactor)

In [ ]:
predict_and_evaluate(teacher, name="TEACHER (BERT-Large)", test_loader=test_dl)
predict_and_evaluate(student, name="STUDENT (Distilled BERT)", test_loader=test_dl)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

teacher_id = "microsoft/phi-2"
student_id = "microsoft/phi-1_5"

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)

if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

teacher = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    device_map="auto",
    quantization_config=quantization_config,
)

for p in list(teacher.parameters())[:5]:
  print(p)

In [ ]:
from transformers import BitsAndBytesConfig

teacher.eval()
for p in teacher.parameters():
  p.requires_grad = False

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

student = AutoModelForCausalLM.from_pretrained(
    student_id,
    device_map="auto",
    quantization_config=quantization_config,
)

prompts = [
    "Explain why the sky is blue. ### The sky appears blue because molecules in Earth's atmosphere scatter sunlight, and blue light is scattered more than other colors due to its shorter wavelength.",
    "What is the capital of France? ### The capital of France is Paris.",
    "Write a short story about a robot and a cat. ### Once upon a time, a lonely robot found a stray cat. They became best friends, exploring the city together, and the robot learned the meaning of companionship."
]

for prompt in prompts:
  teacher_inputs = teacher_tokenizer(prompt, return_tensors="pt", padding=True).to(teacher.device)
  student_inputs = student_tokenizer(prompt, return_tensors="pt", padding=True).to(student.device)

  with torch.no_grad():
    teacher_logits = teacher(**teacher_inputs).logits[:, :-1, :]
    teacher_soft = torch.softmax(teacher_logits/temperature, dim=-1)
    teacher_soft = torch.clamp(teacher_soft, min=1e-8)

  student_logits = student(**student_inputs).logits[:, :-1, :]
  student_logits_soft = torch.log_softmax(student_logits/temperature, dim=-1)
  labels = student_inputs["input_ids"][:, 1:].contiguous()

  loss_soft = kl_loss(student_logits_soft, teacher_soft) * (temperature**2)
  loss_hard = ce_loss(student_logits.reshape(-1, student_logits.shape[-1]), labels.reshape(-1))
  loss = alpha_soft * loss_soft + (1-alpha_soft) * loss_hard

  if torch.isnan(loss):
    print(" NaN detected on prompt:", prompt[:50])
    continue

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  print(f"Prompt: {prompt[:40]}..., Loss: {loss.item():.4f}")

In [ ]:
student.save_pretrained("distilled_phi1_5")
student_tokenizer.save_pretrained("distilled_phi1_5")
print("Distillation complete. Model saved in 'distilled_phi1_5/'")

**LLAMA factory**

In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git

In [ ]:
!pwd

In [ ]:
%cd /content/LLaMA-Factory

In [ ]:
!pwd

In [ ]:
!ls

In [ ]:
!pip install -r requirements.txt
!pip install bitsandbytes>=0.39.0
%cd /content/LLaMA-Factory

In [ ]:
!pwd

In [ ]:
!ls

In [ ]:
!pip install -e .

In [ ]:
!ls /content/LLaMA-Factory/src

In [ ]:
import os
from llamafactory.webui.interface import create_ui
from peft import PeftModel

os.environ["GRADI0_SHARE"] = "i"

In [ ]:
!python /content/LLaMA-Factory/src/webui.py

In [ ]:
ui = create_ui()
ui.launch(share=True)

In [ ]:
base_model = "google/gemma-1.1-2b-it"
lora_path = "/content/LLaMA-Factory/saves/Gemma-1.1-2B-Instruct/lora/train_2025-12-11-17-40-51"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, lora_path)

In [ ]:
model.eval()
prompt = "Can you tell what ls -l would display?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
%cd /content/LLaMA-Factory

In [ ]:
!pwd

In [ ]:
!nvidia-smi

In [ ]:
%env CUDA_LAUNCH_BLOCKING=1

In [ ]:
!python -m llamafactory.cli train train_gemma_qlora.yaml

In [ ]:
!ls output/my-gemma-qlora

In [ ]:
!ls output/test-lora

In [ ]:
base = "google/gemma-1.1-2b-it"
adapter = "/content/LLaMA-Factory/gemma_lora_sft_output"

tokenizer = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, device_map="auto", torch_dtype=torch.float16)

model = PeftModel.from_pretrained(model, adapter)

prompt = "Explain what is QLoRA"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=200)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))
drive.mount('/content/drive')

In [ ]:
!python -m llamafactory.cli export train_gemma_qlora.yaml

### **Unsloth Fine-Tuning**

In [ ]:
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install psutil

In [ ]:
import random
import time, psutil
import numpy as np
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

assert torch.cuda.is_available()

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")

max_seq_length = 4096
dtype = None
load_in_4bit = True

In [ ]:
BASE_MODEL_NAME = "unsloth/tinyllama-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
model, tokenizer

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0.0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407
)

model.print_trainable_parameters()

In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable: {trainable}")
print(f"Total: {total}")
print(f"Percent: {100 * trainable / total:.2f}%")
print(isinstance(model, PeftModel))

for name, param in model.named_parameters():
  if param.requires_grad:
    print(name)

In [ ]:
next(model.parameters()).device
next(model.parameters()).dtype

In [ ]:
model.peft_config
torch.cuda.memory_summary()

In [ ]:
EOS_TOKEN = tokenizer.eos_token

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_data(examples):
  texts = []
  for instruction, input_text, output in zip(
      examples["instruction"], examples["input"], examples["output"]
  ):
    prompt = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
    texts.append(prompt)
  return {"text": texts}

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
dataset = dataset.shuffle(seed=3407).select(range(1500))
dataset = dataset.map(
    format_data,
    batched=True,
    remove_columns=dataset.column_names
)
dataset, dataset["text"][0]

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

process = psutil.Process()
train_start_time = time.time()
cpu_ram_before = process.memory_info().rss / 1024**3

process, train_start_time, cpu_ram_before

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    packing=True,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-5,
        warmup_ratio=0.1,
        optim="adamw_8bit",
        logging_steps=10,
        seed=3407,
        output_dir="outputs",
        report_to="none"
    )
)

trainer.train()
train_end_time = time.time()
cpu_ram_after = process.memory_info().rss / 1024**3

training_time_sec = round(train_end_time - train_start_time, 2)
peak_gpu_vram_gb = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
cpu_ram_used_gb = round(cpu_ram_after - cpu_ram_before, 3)

print("===== UNSLOTH TRAINING STATS =====")
print(f"Training time (sec): {training_time_sec}")
print(f"Peak GPU VRAM (GB): {peak_gpu_vram_gb}")
print(f"CPU RAM used (GB): {cpu_ram_used_gb}")

In [ ]:
FastLanguageModel.for_inference(model)
prompt = alpaca_prompt.format(
    "Continue the Fibonacci sequence",
    "1, 1, 2, 3, 5, 8",
    ""
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
  outputs = model.generate(
      **inputs,
      max_new_tokens=64,
      use_cache=True,
      do_sample=False
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
LORA_SAVE_PATH = "lora_model"

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

print(f"LoRA adapters saved at: {LORA_SAVE_PATH}")

### **Axoletl Tutorial**

In [ ]:
!pip uninstall -y axolotl peft transformers accelerate datasets trl optimum cut-cross-entropy flash-attn
!pip install --no-build-isolation git+https://github.com/OpenAccess-AI-Collective/axolotl.git
!pip install --no-build-isolation axolotl[flash-attn]>=0.9.1
!pip install "cut-cross-entropy[transformers] @ git+https://github.com/axolotl-ai-cloud/ml-cross-entropy.git@318b7e2"

In [ ]:
from axolotl.cli.config import load_cfg
from axolotl.utils.dict import DictDefault
from axolotl.utils import set_pytorch_cuda_alloc_conf
from axolotl.common.datasets import load_datasets
from axolotl.train import train
from transformers import TextStreamer
from huggingface_hub import notebook_login
import os

dataset_id = "winglian/pirate-ultrachat-10k"
uploaded = {}

GOOGLE_DRIVE_PATH = ""
if GOOGLE_DRIVE_PATH:
    from google.colab import drive
    GOOGLE_DRIVE_MNT = "/content/drive/"
    drive.mount(GOOGLE_DRIVE_MNT, force_remount=True)
    tmp_path = os.path.join(GOOGLE_DRIVE_MNT, GOOGLE_DRIVE_PATH.lstrip("/"))
    if not os.path.isfile(tmp_path):
        raise ValueError(f"File {tmp_path} does not exist")
    dataset_id = tmp_path

In [ ]:
os.environ["AXOLOTL_DO_NOT_TRACK"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
config = DictDefault(
    base_model="Qwen/Qwen2.5-3B-Instruct",
    load_in_4bit=True,
    adapter="qlora",
    lora_r=32,
    lora_alpha=64,
    lora_target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "down_proj",
        "up_proj",
    ],
    lora_qkv_kernel=False,
    lora_o_kernel=False,
    lora_mlp_kernel=False,
    embeddings_skip_upcast=True,
    xformers_attention=True,
    plugins=[],
    sample_packing=False,
    learning_rate=0.00019,
    sequence_len=1024,
    micro_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },
    optimizer="paged_adamw_8bit",
    lr_scheduler="cosine",
    warmup_steps=5,
    fp16=True,
    bf16=False,
    max_grad_norm=0.1,
    num_epochs=1,
    saves_per_epoch=2,
    logging_steps=1,
    output_dir="./outputs/qwen-sft-pirate-rrr",
    chat_template="qwen3",
    datasets=[
        {
            "path": dataset_id,
            "type": "chat_template",
            "split": "train",
            "eot_tokens": ["<|im_end|>"],
        }
    ],
    dataloader_prefetch_factor=2,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
)
cfg = load_cfg(config)
set_pytorch_cuda_alloc_conf()
dataset_meta = load_dataset(cfg=cfg)
cfg.max_steps = 25
model, tokenizer, trainer = train(cfg=cfg, dataset_meta=dataset_meta)

In [ ]:
messages = [
  {
    "role": "user",
    "content": "Explain the Pythagorean theorem to me.",
  },
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenizer=False,
    enable_thinking=False
)

outputs = model.generate(
    **tokenizer(prompt, return_tensors="pt").to("cuda"),
    max_new_tokens=192,
    temperature=1.0,
    top_p=0.8,
    top_k=32,
    streamer=TextStreamer(tokenizer, skip_prompt=True)
)

In [ ]:
!ls -lh "./outputs/qwen-sft-pirate-rrr"

In [ ]:
!rm -rf "./outputs/qwen-sft-pirate-rrr/checkpoint-*"

notebook_login()

!huggingface-cli upload --repo-type=model winglian/pirate-qwen-14B "./outputs/qwen-sft-pirate-rrr"